In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
import numpy as np
sys.path.append('../../../')   # Add parent directory to Python path
import pickle
from utils.preprocessing import *
from utils.segmentation import *
from utils.visualization import *

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical


np.random.seed(42)  # For reproducibility


## 1. Combine all datasets for training

### a)Lab data

In [4]:
# Load the curb data (which appears to be stored as a dictionary with scene_0 and scene_1)
with open('../../../data/Training/1s_30hz/raw_data/curb_1s_combined_all.pkl', 'rb') as f:
    curb_data = pickle.load(f)
    data_curb_0 = curb_data['scene_0']
    data_curb_1 = curb_data['scene_1']
    
# Load the other surface types (these appear to be stored as arrays)
with open('../../../data/Training/1s_30hz/raw_data/asphalt_1s_combined_all.pkl', 'rb') as f:
    data_asphalt = pickle.load(f)

with open('../../../data/Training/1s_30hz/raw_data/cobblestone_1s_combined_all.pkl', 'rb') as f:
    data_cobblestone = pickle.load(f)

with open('../../../data/Training/1s_30hz/raw_data/compactgravel_1s_combined_all.pkl', 'rb') as f:
    data_compact_gravel = pickle.load(f)

with open('../../../data/Training/1s_30hz/raw_data/dirt_1s_combined_all.pkl', 'rb') as f:
    data_Dirt = pickle.load(f)

with open('../../../data/Training/1s_30hz/raw_data/pavingstone_1s_combined_all.pkl', 'rb') as f:
    data_PavingStone = pickle.load(f)

# Print shapes to verify the data was loaded correctly
print("Curb (scene 0):", data_curb_0.shape)
print("Curb (scene 1):", data_curb_1.shape)
print("Asphalt:", data_asphalt.shape)
print("Cobblestone:", data_cobblestone.shape)
print("Compact Gravel:", data_compact_gravel.shape)
print("Dirt:", data_Dirt.shape)
print("Paving Stone:", data_PavingStone.shape)


Curb (scene 0): (12267, 30, 3)
Curb (scene 1): (617, 30, 3)
Asphalt: (704, 30, 3)
Cobblestone: (486, 30, 3)
Compact Gravel: (479, 30, 3)
Dirt: (577, 30, 3)
Paving Stone: (645, 30, 3)


### b)Real world data

#### 1_a version a

In [5]:
with open('../../../data/Training/1s_30hz/raw_data/real_world_loso/real_world_1s_combined_5people_unbalanced_testing_version_a.pkl', 'rb') as f:
    data_real_world = pickle.load(f)
    data_real_world_0 = data_real_world['scene_0']
    data_real_world_1 = data_real_world['scene_1']
print("Real World (scene 0):", data_real_world_0.shape)
print("Real World (scene 1):", data_real_world_1.shape)

Real World (scene 0): (18396, 30, 3)
Real World (scene 1): (409, 30, 3)


### 1_b pseodo labels

In [9]:
# load pseudo label data
with open('../../../data/Training/1s_30hz/lab_real_world_2class_unbalanced_version_a/X_test_normalized.pkl', 'rb') as f:
    test_unbalanced_data = pickle.load(f)
with open('../../../data/Training/1s_30hz/lab_real_world_2class_unbalanced_version_a/y_test_predictions_integer.pkl', 'rb') as f:
    test_prediction_unbalanced_labels = pickle.load(f)
    test_prediction_unbalanced_labels_integer = test_prediction_unbalanced_labels['Predicted Label']
test_unbalanced_data.shape, test_prediction_unbalanced_labels_integer.shape

((3708, 30, 3), (3708,))

#### 2) version b

In [4]:
with open('../../../data/Training/1s_30hz/real_world_1s_combined_5people_unbalanced_testing_version_b.pkl', 'rb') as f:
    data_real_world = pickle.load(f)
    data_real_world_0 = data_real_world['scene_0']
    data_real_world_1 = data_real_world['scene_1']
print("Real World (scene 0):", data_real_world_0.shape)
print("Real World (scene 1):", data_real_world_1.shape)

Real World (scene 0): (18627, 30, 3)
Real World (scene 1): (445, 30, 3)


#### 3) version c

In [4]:
with open('../../../data/Training/1s_30hz/real_world_1s_combined_5people_unbalanced_testing_version_c.pkl', 'rb') as f:
    data_real_world = pickle.load(f)
    data_real_world_0 = data_real_world['scene_0']
    data_real_world_1 = data_real_world['scene_1']
print("Real World (scene 0):", data_real_world_0.shape)
print("Real World (scene 1):", data_real_world_1.shape)

Real World (scene 0): (18327, 30, 3)
Real World (scene 1): (432, 30, 3)


#### 4) version d

In [4]:
with open('../../../data/Training/1s_30hz/real_world_1s_combined_5people_unbalanced_testing_version_d.pkl', 'rb') as f:
    data_real_world = pickle.load(f)
    data_real_world_0 = data_real_world['scene_0']
    data_real_world_1 = data_real_world['scene_1']
print("Real World (scene 0):", data_real_world_0.shape)
print("Real World (scene 1):", data_real_world_1.shape)

Real World (scene 0): (18333, 30, 3)
Real World (scene 1): (405, 30, 3)


### 1.1 Balanced binary dataset creation

#### a)Lab data

In [23]:
# Use only curb_1 from original dataset
combined_curb_1 = data_curb_1

# Update the curb_1 data creation
curb_1_data = [(segment, "curb_1") for segment in combined_curb_1]
print(f"Total curb_1 samples: {len(curb_1_data)}")

# Get count of curb_1 samples to know how many we need from other classes
curb_1_count = len(curb_1_data)

# Create list of other datasets
other_datasets = [
    (data_curb_0, "curb_0"),
    (data_asphalt, "asphalt"),
    (data_cobblestone, "cobblestone"),
    (data_compact_gravel, "compact_gravel"),
    (data_Dirt, "dirt"),
    (data_PavingStone, "paving_stone")
]

# Calculate how many samples to take from each other class for even distribution
samples_per_class = curb_1_count // len(other_datasets)
print(f"Taking {samples_per_class} samples from each of the other {len(other_datasets)} classes")

# Randomly select samples from other classes
other_class_data = []
for data, label in other_datasets:
    # Randomly select indices
    selected_indices = np.random.choice(len(data), samples_per_class, replace=False)
    # Add selected samples to other_class_data
    for idx in selected_indices:
        other_class_data.append((data[idx], "non_curb"))  # Label all other classes as "non_curb"

print(f"Total non_curb samples: {len(other_class_data)}")

# Combine curb_1 and other class data
combined_dataset = other_class_data + curb_1_data
print(f"Total binary dataset samples: {len(combined_dataset)}")

Total curb_1 samples: 617
Taking 102 samples from each of the other 6 classes
Total non_curb samples: 612
Total binary dataset samples: 1229


#### b)With real world data

In [24]:
# Combine curb_1 data from both datasets
combined_curb_1 = np.concatenate([data_curb_1, data_real_world_1])

# Update the curb_1 data creation
curb_1_data = [(segment, "curb_1") for segment in combined_curb_1]
print(f"Total combined curb_1 samples: {len(curb_1_data)}")

# Get count of curb_1 samples to know how many we need from other classes
curb_1_count = len(curb_1_data)

# Create list of other datasets
other_datasets = [
    (data_curb_0, "curb_0"),
    (data_asphalt, "asphalt"),
    (data_cobblestone, "cobblestone"),
    (data_compact_gravel, "compact_gravel"),
    (data_Dirt, "dirt"),
    (data_PavingStone, "paving_stone"),
    (data_real_world_0, "real_world_0"),
]

# Calculate how many samples to take from each other class for even distribution
samples_per_class = curb_1_count // len(other_datasets)
print(f"Taking {samples_per_class} samples from each of the other 6 classes")

# Randomly select samples from other classes
other_class_data = []
for data, label in other_datasets:
    # Randomly select indices
    selected_indices = np.random.choice(len(data), samples_per_class, replace=False)
    # Add selected samples to other_class_data
    for idx in selected_indices:
        other_class_data.append((data[idx], "non_curb"))  # Label all other classes as "non_curb"

print(f"Total non_curb samples: {len(other_class_data)}")

# Combine curb_1 and other class data
combined_dataset = other_class_data + curb_1_data
print(f"Total binary dataset samples: {len(combined_dataset)}")

Total combined curb_1 samples: 1022
Taking 146 samples from each of the other 6 classes
Total non_curb samples: 1022
Total binary dataset samples: 2044


### 1.2 Unbalanced binary dataset

#### a)Lab data

In [4]:
# Combine ALL curb_1 data (training datasets only)
combined_curb_1 = data_curb_1

# Combine ALL non-curb data (exclude real_world_0)
combined_non_curb = np.concatenate([
    data_curb_0,
    data_asphalt,
    data_cobblestone,
    data_compact_gravel,
    data_Dirt,
    data_PavingStone
])

print(f"Total curb_1 samples: {len(combined_curb_1)}")
print(f"Total non_curb samples: {len(combined_non_curb)}")

# Also create the combined format if you still need it
curb_1_data = [(segment, "curb_1") for segment in combined_curb_1]
non_curb_data = [(segment, "non_curb") for segment in combined_non_curb]
combined_dataset = non_curb_data + curb_1_data


Total curb_1 samples: 617
Total non_curb samples: 15158


#### b)With Real world data

In [5]:
combined_curb_1 = np.concatenate([data_curb_1, data_real_world_1])

# Combine ALL non-curb data from all datasets
combined_non_curb = np.concatenate([
    data_curb_0,
    data_asphalt,
    data_cobblestone,
    data_compact_gravel,
    data_Dirt,
    data_PavingStone,
    data_real_world_0
])

print(f"Total curb_1 samples: {len(combined_curb_1)}")
print(f"Total non_curb samples: {len(combined_non_curb)}")

# Also create the combined format if you still need it
curb_1_data = [(segment, "curb_1") for segment in combined_curb_1]
non_curb_data = [(segment, "non_curb") for segment in combined_non_curb]

combined_dataset = non_curb_data + curb_1_data

Total curb_1 samples: 1022
Total non_curb samples: 33491


#### c)With Pseodo Real world data

In [10]:
# Split pseudo-labeled data based on predictions
pseudo_curb_1_indices = np.where(test_prediction_unbalanced_labels_integer == 1)[0]
pseudo_non_curb_indices = np.where(test_prediction_unbalanced_labels_integer == 0)[0]

data_pseudo_1 = test_unbalanced_data[pseudo_curb_1_indices]
data_pseudo_0 = test_unbalanced_data[pseudo_non_curb_indices]

print(f"Pseudo-labeled curb_1 samples: {len(data_pseudo_1)}")
print(f"Pseudo-labeled non_curb samples: {len(data_pseudo_0)}")

# Combine curb_1 data from lab, real world, and pseudo-labeled data
combined_curb_1 = np.concatenate([data_curb_1, data_real_world_1, data_pseudo_1])

# Combine ALL non-curb data from all datasets including pseudo-labeled
combined_non_curb = np.concatenate([
    data_curb_0,
    data_asphalt,
    data_cobblestone,
    data_compact_gravel,
    data_Dirt,
    data_PavingStone,
    data_real_world_0,
    data_pseudo_0
])

print(f"\nTotal curb_1 samples (with pseudo): {len(combined_curb_1)}")
print(f"Total non_curb samples (with pseudo): {len(combined_non_curb)}")

# Create the combined format for training
curb_1_data = [(segment, "curb_1") for segment in combined_curb_1]
non_curb_data = [(segment, "non_curb") for segment in combined_non_curb]

combined_dataset = non_curb_data + curb_1_data
print(f"Total binary dataset samples (with pseudo): {len(combined_dataset)}")

Pseudo-labeled curb_1 samples: 449
Pseudo-labeled non_curb samples: 3259

Total curb_1 samples (with pseudo): 1475
Total non_curb samples (with pseudo): 36813
Total binary dataset samples (with pseudo): 38288


## 2. Train Test Spilt

### a) Train, validation spilt

In [12]:
# 80% train, 20% test
train_set, test_set = train_test_split(combined_dataset, test_size=0.2, random_state=42, shuffle=True, stratify=[label for data, label in combined_dataset])
print(len(train_set), len(test_set))
# Separate sensor values and labels
X_train = [x for x, _ in train_set]
y_train = [label for _, label in train_set]
X_val = [x for x, _ in test_set]
y_val = [label for _, label in test_set]
print(f"Label train: {len(y_train)}, Label test: {len(y_val)}")

30630 7658
Label train: 30630, Label test: 7658


### 2.2 Leave one subject out for testing

In [7]:
# leave 1 men out for testing
with open('../../../data/Training/1s_30hz/real_world_1s_scene0_1people_unbalanced_testing_with_curb_activity_version_d.pkl', 'rb') as f:
    test_0 = pickle.load(f)
with open('../../../data/Training/1s_30hz/real_world_1s_scene1_1people_unbalanced_testing_with_curb_activity_version_d.pkl', 'rb') as f:
    test_1 = pickle.load(f)


In [8]:
# Combine test_0 and test_1 segments and labels
X_test = np.concatenate([test_0['segments'], test_1['segments']])
y_test = np.concatenate([
    np.zeros(len(test_0['segments']), dtype=int),  # label 0 for test_0
    np.ones(len(test_1['segments']), dtype=int)    # label 1 for test_1
])

print("Combined test data shape:", X_test.shape)
print("Combined test labels shape:", X_test.shape)
print("Label 0 count:", (y_test == 0).sum())
print("Label 1 count:", (y_test == 1).sum())

Combined test data shape: (3775, 30, 3)
Combined test labels shape: (3775, 30, 3)
Label 0 count: 3669
Label 1 count: 106


## 3. Normalise dataset

### a)Unbalanced dataset

In [13]:
X_train_normalized = normalize_3d_data(X_train)
X_val_normalized = normalize_3d_data(X_val)

In [10]:
X_test_normalized = normalize_3d_data(X_test)

### b) balanced dataset:

In [25]:
# Extract segments and labels separately
X_data = np.array([segment for segment, label in combined_dataset])
y_data = [label for segment, label in combined_dataset]

# Normalize the data
X_normalized = normalize_3d_data(X_data)

## 4. Labels from string to integer

### a)Unbalanced dataset

In [14]:
# Define custom mapping: curb_1=1, non_curb=0
custom_mapping = {"curb_1": 1, "non_curb": 0}

# Apply the custom mapping
y_train_int = np.array([custom_mapping[label] for label in y_train])
y_val_int = np.array([custom_mapping[label] for label in y_val])

# Create and manually adjust the label_encoder to match your encoding
label_encoder = LabelEncoder()
label_encoder.classes_ = np.array(["non_curb", "curb_1"])  # Ensures 0=non_curb, 1=curb_1

print("Classes:", label_encoder.classes_)
print("First 10 y_train_int:", y_train_int[:10])
print("First 10 y_val_int:", y_val_int[:10])
for idx, label in enumerate(label_encoder.classes_):
    print(f"{idx}: {label}")

Classes: ['non_curb' 'curb_1']
First 10 y_train_int: [0 0 0 0 0 0 0 0 0 0]
First 10 y_val_int: [0 0 0 0 0 0 0 0 0 0]
0: non_curb
1: curb_1


## 5: One-hot encode the labels

### a)Normal

In [15]:
y_train_onehot = to_categorical(y_train_int)
y_val_onehot = to_categorical(y_val_int)

print(y_train_onehot.shape)
print(y_val_onehot.shape)

(30630, 2)
(7658, 2)


In [16]:
# Randomly select an index and check that the one-hot encoding matches the original label
r = np.random.randint(len(y_train_int))
assert y_train_onehot[r].argmax() == y_train_int[r]
r = np.random.randint(len(y_val_int))
assert y_val_onehot[r].argmax() == y_val_int[r]

### b)LOSO

In [16]:
# One-hot encode y_test_loso
y_test_onehot = to_categorical(y_test)

print("y_test_loso_onehot shape:", y_test_onehot.shape)
print("First 5 one-hot labels:", y_test_onehot[:5])

y_test_loso_onehot shape: (3775, 2)
First 5 one-hot labels: [[1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]]


## 6. Save train, test data and labels

### a)Balanced dataset

In [27]:
# Save train data
with open('../../../data/Training/1s_30hz/lab_real_world_2class_balanced_version_d/X_train_normalized.pkl', 'wb') as f:
    pickle.dump(X_normalized, f)

### b)Unbalanced dataset

In [17]:
# Save train data
with open('../../../data/Training/1s_30hz/lab_real_world_2class_unbalanced_version_a_pseudo/X_train_normalized.pkl', 'wb') as f:
    pickle.dump(X_train_normalized, f)
with open('../../../data/Training/1s_30hz/lab_real_world_2class_unbalanced_version_a_pseudo/y_train_onehot.pkl', 'wb') as f:
    pickle.dump(y_train_onehot, f)
# Save validation data
with open('../../../data/Training/1s_30hz/lab_real_world_2class_unbalanced_version_a_pseudo/X_val_normalized.pkl', 'wb') as f:
    pickle.dump(X_val_normalized, f)
with open('../../../data/Training/1s_30hz/lab_real_world_2class_unbalanced_version_a_pseudo/y_val_onehot.pkl', 'wb') as f:
    pickle.dump(y_val_onehot, f)

### c)LOSO test data

In [18]:
with open('../../../data/Training/1s_30hz/lab_real_world_2class_unbalanced_version_d/X_test_normalized.pkl', 'wb') as f:
    pickle.dump(X_test_normalized, f)

with open('../../../data/Training/1s_30hz/lab_real_world_2class_unbalanced_version_d/y_test_onehot.pkl', 'wb') as f:
    pickle.dump(y_test_onehot, f)

## 6. Train, validation Spilt

In [11]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_normalized,
    y_train_onehot,
    test_size=0.2,           # 20% for validation
    random_state=42,
    shuffle=True
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)

Train shape: (10096, 30, 3) (10096, 2)
Validation shape: (2524, 30, 3) (2524, 2)


In [15]:
# Save training and validation data
with open('../../../data/Training/1s_30hz/2class_unbalanced/X_train_normalized.pkl', 'wb') as f:
    pickle.dump(X_train, f)

with open('../../../data/Training/1s_30hz/2class_unbalanced/X_val_normalized.pkl', 'wb') as f:
    pickle.dump(X_val, f)

with open('../../../data/Training/1s_30hz/2class_unbalanced/y_train_onehot.pkl', 'wb') as f:
    pickle.dump(y_train, f)

with open('../../../data/Training/1s_30hz/2class_unbalanced/y_val_onehot.pkl', 'wb') as f:
    pickle.dump(y_val, f)